In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added to Python path:")
print(PROJECT_ROOT)

In [ ]:
import MetaTrader5 as mt5

from src.config import CONFIG
from src.loaders import load_mt5_live, fetch_mt5_history
from src.signals import SignalResult, generate_signal, regime_gate, apply_filters
from src.live_runner import run_live_with_context
from src.exports import save_artifacts
from src.mql5_overlay import write_mql5_double_overlay
from src.engine import EngineState, update_engine_state

### Double-overlay MT5 indicator

This version generates the MT5 indicator for the double-overlay workflow.

Use:
- `write_mql5_overlay(...)` for the original straight execution overlay
- `write_mql5_double_overlay(...)` for straight execution + bendy context overlay

In [ ]:
# ── Context overlay: generate MT5 indicator source for double overlay ──
mq5_path = write_mql5_double_overlay(output_path="live_artifacts/exports/VWAP_Overlay.mq5")
print(f"✅ Double-overlay MQL5 source written → {mq5_path}")
print("Now open the generated .mq5 file in MetaEditor, compile it, and attach it to your chart.")

### Context overlay live export prototype

This version keeps the original live runner unchanged and adds a second live
runner that also exports a rolling bendy context-overlay trail to
`live_artifacts/live_context.json`.

Use:
- `run_live(...)` for the original straight execution overlay only
- `run_live_with_context(...)` for straight execution + bendy context overlay

### Double-overlay live runner

Use this version when you want both overlays at the same time:

- the original straight execution overlay from `live_state.json`
- the rolling bendy context overlay from `live_context.json`

Run this instead of the original `run_live(...)` cell.

In [ ]:
# LIVE MODE — double overlay version
# MT5 must be open and logged in before running this

CONFIG['instrument'] = 'US100.cash'

if 'prob_table' not in dir():
    print("❌ prob_table not found — load/export the probability table first")
    raise SystemExit

mt5.shutdown()
if not mt5.initialize():
    print(f"❌ MT5 initialize() failed — error: {mt5.last_error()}")
    print("   Make sure MT5 is open, logged in, and Algo Trading is enabled")
    print("   Tools → Options → Expert Advisors → tick 'Allow algorithmic trading'")
    raise SystemExit

print(f"✅ MT5 connected — {mt5.terminal_info().name}")
print(f"   Account : {mt5.account_info().login} | Server: {mt5.account_info().server}")

symbol_info = mt5.symbol_info(CONFIG['instrument'])

if symbol_info is None:
    print(f"\n❌ Symbol '{CONFIG['instrument']}' not found on this account.")
    print("   Symbols on this broker containing US/NAS/TECH:")
    all_syms = mt5.symbols_get()
    matches = [s.name for s in all_syms
               if any(x in s.name.upper() for x in ['US1', 'NAS', 'TECH', 'NDX', 'USTEC'])]
    print(f"   {matches[:20]}")
    print("\n   Update CONFIG['instrument'] to the correct name above, then re-run.")
    mt5.shutdown()
    raise SystemExit

live_spread = symbol_info.ask - symbol_info.bid
print(f"\n✅ Symbol found: {symbol_info.name}")
print(f"   Bid: {symbol_info.bid} | Ask: {symbol_info.ask}")
print(f"   Live spread right now: {live_spread:.1f} points")
print(f"   ⚠️ If this looks right, set CONFIG['spread_cost'] = {live_spread:.1f} and re-run relevant calibration cells")
print("   (only matters for outcome labelling accuracy, not for live signal generation)")

print("\n🟢 Starting live engine with double overlay — press the ■ stop button in Jupyter to stop\n")

try:
    run_live_with_context(
        symbol=CONFIG['instrument'],
        timeframe_mt5=mt5.TIMEFRAME_M1,
        config=CONFIG,
        prob_table=prob_table,
        marginal_table=prob_table,
        load_mt5_live=load_mt5_live,
        EngineState=EngineState,
        update_engine_state=update_engine_state,
        generate_signal=generate_signal,
        regime_gate=regime_gate,
        apply_filters=apply_filters,
    )
except KeyboardInterrupt:
    print("\n🔴 Live run stopped by user")
finally:
    mt5.shutdown()
    print("🔌 MT5 connection closed")